# 검증 데이터 분석: 안전모 미착용

두 모델의 val 결과를 확인하고 탐지 기준값과 오류 사례를 살펴봅니다. test는 사용하지 않습니다.


In [ ]:
!pip -q install ultralytics==8.4.208
import json, zipfile, yaml, torch
from pathlib import Path
from ultralytics import YOLO
from google.colab import drive
assert torch.cuda.is_available(), "GPU 런타임을 선택하세요."
drive.mount("/content/drive")


In [ ]:
ROOT = Path("/content/drive/MyDrive/ppe-project")
ZIP = Path("/content/drive/MyDrive/construction-ppe-5class.zip")
DATA = Path("/content/ppe_project_data/construction-ppe-5class")
assert ZIP.is_file(), f"ZIP을 찾지 못했습니다: {ZIP}"
if not (DATA / "split_manifest.csv").is_file():
    DATA.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP) as z:
        assert z.testzip() is None
        z.extractall(DATA.parent)
config = yaml.safe_load((DATA / "data.yaml").read_text())
config["path"] = str(DATA)
YAML = DATA / "data_colab.yaml"
YAML.write_text(yaml.safe_dump(config, sort_keys=False))
assert len(list((DATA / "images/val").glob("*.jpg"))) == 100
WEIGHTS = {"yolo11n": ROOT / "runs/baseline_yolo11n_seed42/weights/best.pt", "yolo26n": ROOT / "runs/proposed_yolo26n_seed42/weights/best.pt"}
for name, weight in WEIGHTS.items():
    assert weight.is_file(), f"가중치가 없습니다: {weight}"
    print(name, weight)


## 검증 결과 및 예측 이미지 저장

각 모델의 val 예측과 혼동행렬, precision-recall 곡선을 저장합니다. 기준값 0.001에서 곡선을 계산하며, 이후 예측 이미지는 기준값 0.25의 확인용 예시입니다. 미탐 여부는 정답 라벨과 비교해 확인합니다.


In [ ]:
OUT = ROOT / "val_analysis"
OUT.mkdir(parents=True, exist_ok=True)
summary = {}
for name, weight in WEIGHTS.items():
    run = OUT / (name + "_metrics")
    if run.exists():
        raise FileExistsError(f"기존 결과가 있습니다: {run}")
    metrics = YOLO(str(weight)).val(data=str(YAML), split="val", imgsz=640, batch=8, device=0, conf=0.001, plots=True, project=str(OUT), name=name+"_metrics", exist_ok=False)
    box = metrics.box
    summary[name] = {"precision":float(box.mp), "recall":float(box.mr), "mAP50":float(box.map50), "mAP50_95":float(box.map), "classes":{str(metrics.names[int(c)]):{"precision":float(box.p[i]), "recall":float(box.r[i]), "AP50":float(box.ap50[i])} for i,c in enumerate(box.ap_class_index)}}
    (OUT / (name+"_summary.json")).write_text(json.dumps(summary[name],ensure_ascii=False,indent=2))
print(json.dumps(summary,ensure_ascii=False,indent=2))


## 안전모 미착용 정답이 있는 이미지 예측

val에서 `no_helmet` 정답이 있는 이미지만 고르고, 두 모델의 예측 이미지를 Drive에 저장합니다. 정답과 비교하여 미탐 여부를 판정해야 합니다. 전체 이미지 저장에는 시간이 조금 걸립니다.


In [ ]:
images=[]
for label in sorted((DATA / "labels/val").glob("*.txt")):
    if any(line.split() and line.split()[0] == "2" for line in label.read_text().splitlines()):
        image=DATA / "images/val" / (label.stem+".jpg")
        if image.is_file(): images.append(str(image))
print("no_helmet 정답 포함 이미지:", len(images))
for name, weight in WEIGHTS.items():
    run=OUT / (name+"_no_helmet_examples")
    if run.exists(): raise FileExistsError(f"기존 결과가 있습니다: {run}")
    YOLO(str(weight)).predict(source=images, conf=0.25, imgsz=640, device=0, save=True, project=str(OUT), name=name+"_no_helmet_examples", exist_ok=False, verbose=False)
print("결과 폴더:", OUT)
